### Étape 1 : Installer les dépendances

Nous installons ici les paquets et bibliothèques nécessaires au système :

- **llama-index** : utilisé pour la recherche de documents, la gestion des embeddings et la construction d’index.
- **langchain** : permet de créer et d’orchestrer des agents, des outils et des chaînes de traitement.
- **langchain_community** : nécessaire pour intégrer `ChatOpenAI` avec LangChain (version 0.3.x).
- **openai** : donne accès à l’API des modèles LLM.
- **wikipedia** : outil optionnel permettant à l’agent de récupérer des informations directement depuis Wikipédia.


In [1]:
# ============================================================
# INSTALLATION DES DEPENDANCES
# ------------------------------------------------------------
# NOTE (corrige un bug d'environnement) : llama-index==0.9.41 est une
# version ancienne (2023) qui utilise en interne le module pkg_resources.
# Les versions recentes de setuptools (>= 81) ont supprime pkg_resources,
# ce qui fait planter l'import de llama_index avec
# "ModuleNotFoundError: No module named 'pkg_resources'".
# On epingle donc setuptools<81 pour garder pkg_resources disponible.
# ============================================================

!pip install --upgrade pip
!pip install "setuptools<81"
!pip install llama-index==0.9.41 langchain==0.3.27 langchain_community openai==1.101.0 reportlab python-dotenv


Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


Defaulting to user installation because normal site-packages is not writeable


1. **Requête utilisateur**  
   Tout commence par une question de l’utilisateur. La requête est transmise au composant central : **l’agent**.

2. **L'agent**  
   L'agent joue le rôle de « cerveau » du système. Il analyse la requête et détermine quel outil spécialisé doit traiter les différentes parties.  
   Contrairement à un pipeline figé, l’agent **prend des décisions dynamiques** selon les besoins de la requête.

3. **Prise de décision et outils**  
   En fonction de la requête, l’agent peut choisir parmi plusieurs outils :

   - **Outil DocumentRetriever** : trouve et récupère les documents pertinents pour enrichir le contexte.  
   - **Outil Calculatrice** : résout les problèmes mathématiques ou les calculs complexes.  
   - **Outil Wikipédia** : recherche des informations factuelles directement sur Wikipédia.

   L'agent peut appeler un outil **une ou plusieurs fois** ou **combiner plusieurs outils** selon la complexité de la tâche.
   4. **Moteur de requêtes LlamaIndex**  
   Certains outils, comme **DocumentRetriever** ou **Calculator Tool**, transmettent leurs résultats au **moteur de requêtes LlamaIndex** (un moteur de recherche et de synthèse spécialisé).  
   
   LlamaIndex traite, fusionne et structure les informations fournies par ces outils afin de produire une réponse cohérente, détaillée et précise.

5. **Résultat final**  
   Une fois que l’agent estime que les informations sont suffisantes et correctement combinées, il renvoie la réponse finale à l’utilisateur.

---

### **Remarque**  
Contrairement à un pipeline traditionnel, ce système permet à l’agent de prendre des **décisions intelligentes, dynamiques et contextuelles** quant aux outils, aux sources de données et au moment optimal pour les utiliser.  
Cela imite un véritable **processus de raisonnement et de planification**, faisant de l’ensemble un **système agentique avancé**.



In [2]:
# Import des briques LlamaIndex (recherche documentaire / index vectoriel)
from llama_index import SimpleDirectoryReader, GPTVectorStoreIndex, LLMPredictor, ServiceContext
# Import du wrapper LangChain autour des modeles de chat OpenAI
from langchain.chat_models import ChatOpenAI
# Import des briques LangChain pour construire un agent (outils, initialisation, type d'agent)
from langchain.agents import Tool, initialize_agent, AgentType
import os  # acces aux variables d'environnement (pour la cle API, notamment)

# NOTE : les avertissements "LangChainDeprecationWarning" affiches a l'import
# sont normaux avec ces versions de bibliotheques : ce sont des imports
# encore valides mais marques comme obsoletes par LangChain. Ils n'empechent
# pas le notebook de fonctionner.


C:\Users\axiat\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\llama_index\download\module.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [3]:
# ============================================================
# CONFIGURATION DE LA CLE API OPENAI
# ------------------------------------------------------------
# ATTENTION SECURITE (corrige une fuite de cle API) :
# La version originale de cette cellule contenait une vraie cle API OpenAI
# ecrite EN CLAIR dans le code ("sk-proj-..."). Une cle ecrite ainsi finit
# presque toujours par etre commitee dans un depot git, partagee par erreur,
# ou lue par n'importe qui ayant acces au fichier -> c'est une fuite de
# credentials. Si cette cle a deja circule quelque part, il faut la revoquer
# immediatement sur https://platform.openai.com/api-keys et en generer une
# nouvelle.
#
# Regle a retenir : une cle API ne doit JAMAIS etre ecrite en dur dans un
# notebook ou un script. On la lit depuis un fichier ".env" local, qui reste
# sur ta machine et n'est jamais commite (voir .gitignore).
#
# MISE EN PLACE (a faire une seule fois) :
#   1. copie le fichier ".env.example" (a la racine du projet) en ".env"
#   2. ouvre ".env" et remplace la valeur par ta vraie cle OpenAI
#   3. ne modifie jamais .env.example avec une vraie cle (lui reste commite/partage)
#
# MODE MOCK / MODE REEL :
# Si aucune cle n'est definie (pas de fichier .env, ou variable vide), ce
# notebook bascule automatiquement en "mode mock" : les appels au LLM sont
# simules (reponses pre-ecrites, aucun appel reseau, aucun cout). Cela
# permet de tester tout le pipeline (chargement de documents, indexation,
# agent, outils) sans avoir besoin d'une cle valide. Des qu'une cle est
# fournie via .env, le notebook utilise le vrai modele OpenAI (gpt-3.5-turbo).
# ============================================================

from dotenv import load_dotenv  # charge les variables definies dans un fichier .env

# Cherche un fichier ".env" dans le dossier courant (ou un dossier parent) et
# charge son contenu dans les variables d'environnement du processus Python.
# Si aucun fichier .env n'existe, load_dotenv() ne fait rien (pas d'erreur).
load_dotenv()

# Lit la cle une fois chargee par load_dotenv() ci-dessus.
cle_api = os.environ.get("OPENAI_API_KEY", "").strip()

# MODE_REEL = True si une cle a ete trouvee -> on utilisera le vrai LLM OpenAI.
# MODE_REEL = False -> on utilisera des reponses simulees (voir cellules suivantes).
MODE_REEL = bool(cle_api)

if MODE_REEL:
    print("Cle OPENAI_API_KEY detectee -> MODE REEL (vrais appels a l'API OpenAI, factures).")
else:
    print("Aucune cle OPENAI_API_KEY definie -> MODE MOCK (reponses simulees, aucun appel reseau/facture).")
    print("Pour activer le mode reel : copie .env.example en .env, renseigne ta cle,")
    print("puis redemarre le kernel et relance le notebook depuis le debut.")


Aucune cle OPENAI_API_KEY definie -> MODE MOCK (reponses simulees, aucun appel reseau/facture).
Pour activer le mode reel : copie .env.example en .env, renseigne ta cle,
puis redemarre le kernel et relance le notebook depuis le debut.


In [4]:
# ============================================================
# DOSSIER DE DOCUMENTS (remplace Google Drive / Google Colab)
# ------------------------------------------------------------
# NOTE (corrige un bug d'environnement) : la version originale utilisait
# `google.colab.drive.mount(...)`, qui ne fonctionne QUE dans Google Colab
# et plante immediatement (ModuleNotFoundError) sur un Jupyter local. On
# utilise a la place un simple dossier local "documents/", cree s'il
# n'existe pas encore.
# ============================================================

import os  # (deja importe plus haut, reimporte ici pour que la cellule soit autonome)

DOSSIER_DOCUMENTS = "documents"        # dossier local contenant les PDF a indexer
os.makedirs(DOSSIER_DOCUMENTS, exist_ok=True)   # cree le dossier s'il n'existe pas

print(f"Dossier de documents pret : ./{DOSSIER_DOCUMENTS}/")
print("Depose tes propres PDF dans ce dossier si tu veux indexer tes vrais documents.")


Dossier de documents pret : ./documents/
Depose tes propres PDF dans ce dossier si tu veux indexer tes vrais documents.


In [5]:
# ============================================================
# DOCUMENT D'EXEMPLE (remplace google.colab.files.upload)
# ------------------------------------------------------------
# NOTE (corrige un bug d'environnement) : la version originale utilisait
# `google.colab.files.upload()`, une boite de dialogue d'upload qui
# n'existe que dans l'interface web de Google Colab -> plante ailleurs.
#
# Ici, si le dossier "documents/" est vide, on genere automatiquement un
# petit PDF d'exemple (avec la bibliotheque reportlab) afin que le reste
# du notebook (indexation, recherche) ait toujours quelque chose a
# traiter, meme sans document fourni par l'utilisateur.
# ============================================================

import os

chemin_pdf_exemple = os.path.join(DOSSIER_DOCUMENTS, "exemple_rapport.pdf")

# On ne regenere le PDF que si le dossier est vide (ne pas ecraser les vrais documents de l'utilisateur)
fichiers_existants = [f for f in os.listdir(DOSSIER_DOCUMENTS) if f.lower().endswith(".pdf")]

if not fichiers_existants:
    from reportlab.lib.pagesizes import A4
    from reportlab.pdfgen import canvas

    # Contenu fictif (aucune donnee reelle), juste pour valider le pipeline RAG
    lignes = [
        "Rapport Exemple - Societe Fictive SA",
        "",
        "Ce document est un exemple genere automatiquement pour tester le pipeline RAG.",
        "La societe fictive a un capital de 500 millions d'euros.",
        "Le ratio de solvabilite (CET1) s'etablit a 15.2 pourcent au dernier trimestre.",
        "Les risques principaux identifies sont le risque de credit, le risque de marche",
        "et le risque operationnel.",
    ]

    c = canvas.Canvas(chemin_pdf_exemple, pagesize=A4)
    y = 800                              # position verticale de depart (haut de page)
    for ligne in lignes:
        c.drawString(50, y, ligne)        # ecrit une ligne de texte a (x=50, y)
        y -= 20                           # descend de 20 points pour la ligne suivante
    c.save()

    print(f"Aucun PDF trouve -> document d'exemple genere : {chemin_pdf_exemple}")
else:
    print(f"Documents deja presents dans {DOSSIER_DOCUMENTS}/ : {fichiers_existants}")

print("Contenu du dossier documents :", os.listdir(DOSSIER_DOCUMENTS))


Documents deja presents dans documents/ : ['exemple_rapport.pdf']
Contenu du dossier documents : ['exemple_rapport.pdf']


In [6]:
!pip install wikipedia


Defaulting to user installation because normal site-packages is not writeable


### Importation des bibliothèques nécessaires

Nous importons ensuite les modules indispensables au fonctionnement du système :

- **SimpleDirectoryReader** : charge les documents depuis un dossier afin de les indexer.
- **GPTVectorStoreIndex** : crée un index vectoriel pour la recherche d'information basée sur les embeddings.
- **LLMPredictor** et **ServiceContext** : servent d’enveloppe LLM et de contexte d’exécution pour LlamaIndex.
- **ChatOpenAI** : modèle GPT d’OpenAI utilisé pour la génération de texte et les tâches cognitives.
- **Tool**, **initialize_agent**, **AgentType** : nécessaires pour construire un agent capable de raisonner et d'appeler dynamiquement des outils.
- **ConversationBufferMemory** : conserve l’historique des échanges afin d’améliorer la cohérence de l’agent au fil des interactions.
- **wikipedia** : outil permettant d’interroger Wikipédia pour obtenir des connaissances générales extérieures au système.


In [7]:
# Import de la classe LlamaIndex necessaire pour construire l'index vectoriel
from llama_index import SimpleDirectoryReader, GPTVectorStoreIndex, LLMPredictor, ServiceContext
from langchain.chat_models import ChatOpenAI                       # modele de chat OpenAI (mode reel)
from langchain.agents import Tool, initialize_agent, AgentType     # briques pour construire l'agent
from langchain.agents import AgentExecutor                          # (non utilise directement, garde pour compatibilite)
from langchain.memory import ConversationBufferMemory               # memoire de conversation de l'agent
import wikipedia                                                     # acces a l'API de recherche Wikipedia


In [8]:
!pip install pypdf


Defaulting to user installation because normal site-packages is not writeable


### Construction du système de recherche LlamaIndex

Nous mettons en place l’architecture de recherche basée sur LlamaIndex :

- **SimpleDirectoryReader** : lit automatiquement tous les documents présents dans le dossier fourni.
- **LLMPredictor** : encapsule le modèle `gpt-3.5-turbo` pour l’utiliser comme moteur de génération et d’analyse au sein de LlamaIndex.
- **GPTVectorStoreIndex** : transforme les documents en **vecteurs** (embeddings) et les stocke dans une base vectorielle optimisée pour la recherche sémantique.
- **Moteur de requêtes** : à chaque requête, il renvoie les **3 documents les plus pertinents**, sur base de leur similarité vectorielle.


In [9]:
# ============================================================
# CONSTRUCTION DE L'INDEX DE RECHERCHE (LlamaIndex)
# ------------------------------------------------------------
# NOTE (corrige plusieurs bugs) :
# 1. Le chemin du PDF pointait en dur vers Google Drive
#    ("/content/drive/MyDrive/AGENTIC/..."). On utilise maintenant tous
#    les PDF presents dans le dossier local DOSSIER_DOCUMENTS (defini en
#    cellule 5/6).
# 2. En MODE MOCK (pas de cle API), on utilise MockLLM et MockEmbedding
#    (fournis par LlamaIndex lui-meme pour les tests) afin de construire
#    et d'interroger un vrai index, sans appeler l'API OpenAI. En MODE
#    REEL, on utilise le vrai modele de chat + les embeddings OpenAI par
#    defaut de LlamaIndex.
# ============================================================

import glob

# Recupere tous les PDF du dossier de documents (au moins celui genere en cellule 6)
chemins_pdf = glob.glob(os.path.join(DOSSIER_DOCUMENTS, "*.pdf"))
assert chemins_pdf, f"Aucun PDF trouve dans {DOSSIER_DOCUMENTS}/ ! Depose un fichier .pdf dans ce dossier."

# Etape 1 : Charger les documents PDF (extraction du texte, page par page)
documents = SimpleDirectoryReader(input_files=chemins_pdf).load_data()
print(f"{len(documents)} document(s) charge(s) depuis {DOSSIER_DOCUMENTS}/")

if MODE_REEL:
    # ---- MODE REEL : vrai LLM OpenAI + vrais embeddings OpenAI ----
    llm_predictor = LLMPredictor(llm=ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo"))
    service_context = ServiceContext.from_defaults(llm_predictor=llm_predictor)
else:
    # ---- MODE MOCK : LLM et embeddings simules (fournis par LlamaIndex) ----
    from llama_index import MockEmbedding
    from llama_index.llms.mock import MockLLM

    llm_predictor = LLMPredictor(llm=MockLLM())                  # LLM simule : ne genere pas de vrai texte
    service_context = ServiceContext.from_defaults(
        llm_predictor=llm_predictor,
        embed_model=MockEmbedding(embed_dim=256)                  # embeddings simules (vecteurs deterministes)
    )
    print("MODE MOCK : l'index est construit avec des embeddings/LLM simules (pas d'appel reseau).")

# Etape 2 : Construire l'index vectoriel a partir des documents charges
index = GPTVectorStoreIndex.from_documents(documents, service_context=service_context)

# Etape 3 : Creer le moteur de requetes (retourne les 3 passages les plus pertinents)
query_engine = index.as_query_engine(retriever_mode="default", similarity_top_k=3)

print("Index et moteur de requetes prets.")


1 document(s) charge(s) depuis documents/


LLMPredictor is deprecated, please use LLM instead.
MODE MOCK : l'index est construit avec des embeddings/LLM simules (pas d'appel reseau).


Index et moteur de requetes prets.


### Définition des outils à disposition de l’agent

Nous définissons les différents outils que l’agent pourra utiliser :

- **DocumentRetriever** : utilise LlamaIndex pour récupérer les documents les plus pertinents en fonction de la requête.
- **Calculatrice** : gère les requêtes numériques (opérations, calculs, problèmes mathématiques).
- **Wikipédia** : permet de récupérer des connaissances générales qui ne figurent pas dans les documents locaux.
- **Outils** : chacun de ces outils est ensuite rendu **automatiquement accessible** par l’agent, qui peut les appeler dynamiquement selon les besoins de la requête.


In [10]:
import ast       # analyse syntaxique securisee des expressions mathematiques
import operator  # fonctions correspondant aux operateurs (+, -, *, /, ...)

# ---- Outil 1 : recherche documentaire (LlamaIndex) ----
def retrieve_docs(query: str) -> str:
    """Interroge l'index LlamaIndex et retourne la reponse (ou le contexte en mode mock)."""
    response = query_engine.query(query)
    return str(response)


# ---- Outil 2 : calculatrice ----
# NOTE SECURITE (corrige une vulnerabilite) : la version originale utilisait
# `eval(query)` directement sur l'entree fournie a l'agent (donc, indirectement,
# potentiellement influencee par l'utilisateur ou par le LLM). `eval()` execute
# n'importe quel code Python -> un input comme "__import__('os').system('...')"
# permettrait d'executer des commandes arbitraires sur la machine.
# On remplace `eval()` par un evaluateur mathematique restreint, construit sur
# le module `ast` : il n'autorise QUE les nombres et les operateurs
# arithmetiques de base (+ - * / ** % et le signe -), rien d'autre.

_OPERATEURS_AUTORISES = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.FloorDiv: operator.floordiv,
    ast.Mod: operator.mod,
    ast.Pow: operator.pow,
    ast.USub: operator.neg,   # signe moins unaire, ex: -5
    ast.UAdd: operator.pos,   # signe plus unaire, ex: +5
}


def _evaluer_noeud(noeud):
    """Evalue recursivement un noeud de l'arbre syntaxique (AST), en n'acceptant
    que des nombres et des operations arithmetiques (aucun appel de fonction,
    aucune importation, aucun acces a une variable ou un attribut)."""
    if isinstance(noeud, ast.Constant) and isinstance(noeud.value, (int, float)):
        return noeud.value
    if isinstance(noeud, ast.BinOp) and type(noeud.op) in _OPERATEURS_AUTORISES:
        gauche = _evaluer_noeud(noeud.left)
        droite = _evaluer_noeud(noeud.right)
        return _OPERATEURS_AUTORISES[type(noeud.op)](gauche, droite)
    if isinstance(noeud, ast.UnaryOp) and type(noeud.op) in _OPERATEURS_AUTORISES:
        return _OPERATEURS_AUTORISES[type(noeud.op)](_evaluer_noeud(noeud.operand))
    # Tout le reste (appel de fonction, nom de variable, attribut, chaine, etc.) est refuse
    raise ValueError("Expression non autorisee")


def calculator(query: str) -> str:
    """Evalue une expression arithmetique simple de maniere securisee."""
    try:
        arbre = ast.parse(query, mode="eval")     # transforme le texte en arbre syntaxique
        resultat = _evaluer_noeud(arbre.body)      # evalue l'arbre en n'autorisant que l'arithmetique
        return str(resultat)
    except Exception:
        return "Cannot calculate that."


# ---- Outil 3 : recherche Wikipedia ----
def wiki_search(query: str) -> str:
    """Recherche un resume Wikipedia pour la requete donnee."""
    try:
        summary = wikipedia.summary(query, sentences=2)
        return summary
    except Exception:
        # Regroupe toutes les erreurs possibles (page introuvable, ambiguite,
        # pas de reseau, etc.) sous un seul message clair pour l'agent.
        return "No Wikipedia info found."


In [11]:
# Liste des outils mis a disposition de l'agent, chacun avec un nom et une
# description (c'est cette description que le LLM lit pour decider QUAND
# utiliser quel outil).
tools = [
    Tool(name="DocumentRetriever", func=retrieve_docs, description="Use this tool to answer questions using uploaded documents."),
    Tool(name="Calculator", func=calculator, description="Use this tool to solve math expressions."),
    Tool(name="Wikipedia", func=wiki_search, description="Use this tool to get Wikipedia summaries.")
]


### Initialisation de la mémoire de l’agent

Nous initialisons une **mémoire de conversation** pour permettre à l’agent de garder le contexte durant la session :

- **ConversationBufferMemory** : conserve une trace des requêtes et des réponses précédentes, afin que l’agent puisse tenir compte de l’historique dans ses réponses.
- **AgentType.ZERO_SHOT_REACT_DESCRIPTION** : configure un agent capable de décider **quel outil appeler** à partir de la description de la tâche, sans phase de pré-entraînement spécifique.
- **verbose = True** : permet d’afficher les différentes étapes de raisonnement et les appels d’outils dans les logs, ce qui est utile pour le **débogage** et la compréhension du comportement de l’agent.


In [12]:
# ============================================================
# INITIALISATION DE L'AGENT
# ------------------------------------------------------------
# NOTE : en MODE MOCK, on remplace le vrai ChatOpenAI par FakeListChatModel
# (fourni par langchain_community), qui retourne des reponses pre-ecrites
# dans l'ordre, sans aucun appel reseau. Cela permet de tester le
# fonctionnement complet de l'agent ReAct (Thought / Action / Action Input /
# Observation / Final Answer) de bout en bout. Les reponses simulees
# ci-dessous correspondent exactement aux 2 questions de demonstration de
# la derniere cellule du notebook (section "Interaction avec l'agent").
# ============================================================

# Memoire de conversation : permet a l'agent de garder le contexte entre les questions
memory = ConversationBufferMemory(memory_key="chat_history")

if MODE_REEL:
    llm_agent = ChatOpenAI(temperature=0, model_name="gpt-3.5-turbo")
else:
    from langchain_community.chat_models.fake import FakeListChatModel

    # Reponses simulees, au format ReAct attendu par AgentType.ZERO_SHOT_REACT_DESCRIPTION :
    # 2 reponses pour la question "calcul" (Action puis Final Answer),
    # puis 2 reponses pour la question "document" (Action puis Final Answer).
    reponses_simulees = [
        "Thought: Il faut utiliser la calculatrice pour ce calcul.\nAction: Calculator\nAction Input: 12 * 7",
        "Thought: J'ai le resultat du calcul.\nFinal Answer: 12 x 7 = 84.",
        "Thought: Il faut chercher dans les documents fournis.\nAction: DocumentRetriever\nAction Input: ratio de solvabilite",
        "Thought: J'ai trouve l'information dans le document.\nFinal Answer: Voir le contexte du document recupere ci-dessus.",
    ]
    llm_agent = FakeListChatModel(responses=reponses_simulees)

# ZERO_SHOT_REACT_DESCRIPTION : l'agent decide quel outil appeler en se basant
# uniquement sur la description de chaque outil (pas d'exemples fournis).
agent = initialize_agent(
    tools=tools,
    llm=llm_agent,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    memory=memory,
    handle_parsing_errors=True,   # si le LLM produit une sortie mal formee, l'agent essaie de recuperer plutot que de planter
    verbose=True                  # affiche le raisonnement (Thought/Action/Observation) dans les logs
)

print(f"Agent initialise en mode {'REEL' if MODE_REEL else 'MOCK'}.")


Agent initialise en mode MOCK.


C:\Users\axiat\AppData\Local\Temp\ipykernel_38996\2993410349.py:14: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history")
C:\Users\axiat\AppData\Local\Temp\ipykernel_38996\2993410349.py:34: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


### Interaction avec l’agent

- Les utilisateurs peuvent **interagir directement avec l’agent** en lui posant des questions ou en soumettant des requêtes.
- L’agent **choisit automatiquement l’outil le plus adapté**, récupère les informations pertinentes, puis **génère une réponse structurée**.
- Le système prend en charge :
  - les **requêtes documentaires** (via LlamaIndex / DocumentRetriever),
  - les **calculs mathématiques** (via l’outil Calculatrice),
  - la **recherche de connaissances générales** (via Wikipédia).


In [13]:
# ============================================================
# INTERACTION AVEC L'AGENT
# ------------------------------------------------------------
# NOTE (corrige un bug d'execution) : la version originale bouclait
# indefiniment sur `input()`. Lancee de maniere non-interactive (par
# exemple via `jupyter nbconvert --execute`, comme pour les tests
# automatises de ce notebook), `input()` leve une EOFError des le premier
# appel et fait planter tout le notebook. On protege donc l'appel avec
# un `try/except EOFError` pour sortir proprement si aucune entree
# interactive n'est disponible.
#
# En MODE MOCK, une boucle interactive libre n'a pas de sens : le modele
# simule ne connait que les 2 reponses pre-ecrites en cellule 17. On
# lance donc automatiquement les 2 questions de demonstration correspondantes,
# puis on explique comment passer en mode interactif reel.
# ============================================================

print("Agentic RAG active. Ecrire EXIT pour arreter (mode reel uniquement).")

if not MODE_REEL:
    print("\nMODE MOCK : demonstration automatique avec 2 questions scriptees")
    print("(voir les reponses simulees definies en cellule 17).\n")

    questions_demo = [
        "Combien font 12 multiplie par 7 ?",
        "Que dit le document au sujet du ratio de solvabilite ?",
    ]
    for question in questions_demo:
        print(f"You: {question}")
        response = agent.run(question)
        print("Agent:", response)
        print()

    print("Pour discuter librement avec l'agent (vraies reponses du LLM) :")
    print("  1. copie .env.example en .env et renseigne une cle OPENAI_API_KEY valide")
    print("  2. redemarre le kernel et relance le notebook depuis le debut")
    print("  3. relance cette cellule : la boucle interactive ci-dessous s'activera")
else:
    while True:
        try:
            query = input("You: ")
        except EOFError:
            # Pas d'entree interactive disponible (execution automatisee) -> on sort proprement
            print("(entree non interactive detectee, arret de la boucle)")
            break

        if query.strip().lower() in ["exit", "quit"]:
            break
        if not query.strip():
            continue

        response = agent.run(query)
        print("Agent:", response)


Agentic RAG active. Ecrire EXIT pour arreter (mode reel uniquement).

MODE MOCK : demonstration automatique avec 2 questions scriptees
(voir les reponses simulees definies en cellule 17).

You: Combien font 12 multiplie par 7 ?


> Entering new AgentExecutor chain...
Thought: Il faut utiliser la calculatrice pour ce calcul.
Action: Calculator
Action Input: 12 * 7
Observation: 84
Thought:Thought: J'ai le resultat du calcul.
Final Answer: 12 x 7 = 84.

> Finished chain.
Agent: 12 x 7 = 84.

You: Que dit le document au sujet du ratio de solvabilite ?


> Entering new AgentExecutor chain...
Thought: Il faut chercher dans les documents fournis.
Action: DocumentRetriever
Action Input: ratio de solvabilite
Observation: Context information is below.
---------------------
page_label: 1
file_path: documents\exemple_rapport.pdf

Rapport Exemple - Societe Fictive SA
Ce document est un exemple genere automatiquement pour tester le pipeline RAG.
La societe fictive a un capital de 500 millions d'euro

C:\Users\axiat\AppData\Local\Temp\ipykernel_38996\755169476.py:30: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = agent.run(question)
